# 7-class vehicle classifier (A/C/D/E/G/V) — Colab GPU

Fixes the class problem in the ITS pipeline: base COCO has four vehicle classes
and the mentor taxonomy has seven, so **C** (light truck) and **V** (van) cannot
be expressed and are guessed from a monocular size heuristic. Worse on the target
footage, COCO has no **micro-bus** concept at all — and microbuses are about a
third of the traffic on `street_egypt.mp4`.

**Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.
About 45 minutes: ~15 downloading, ~27 training.

### Measured results (v1, 30 epochs)

| | current size heuristic | this model |
|---|---|---|
| A private car | 0.920 | 0.920 |
| E bus/microbus | **0.042** | **0.833** |
| C light truck | 0.526 | 0.158 |
| **overall (68 vehicles)** | **0.500** | **0.676** |

v1 fixed microbuses (1-in-24 correct -> 20-in-24) but regressed on C, because
the Gulf dataset's `lgv` folder (2,229 light-goods-vehicle images) was excluded
as ambiguous. It is not ambiguous — `lgv-van` is a separate class — so `lgv`
maps to **C**. This notebook includes that fix; C trains on 1600 images instead
of 478.

### The one rule

**Judge this on the in-domain test at the bottom, never on val accuracy.** The
val split is Roboflow's own photos. This project has already been burned once:
the EALPR plate detector scored mAP50 **0.985** and was a straight regression on
real footage (`docs/anpr-plan.md` §2d).

In [ ]:
import torch, sys
print('torch', torch.__version__, '| python', sys.version.split()[0])
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!pip -q install ultralytics roboflow
import ultralytics; ultralytics.checks()

## Roboflow key

Store it as a Colab **secret** named `ROBOFLOW_API_KEY` (key icon in the left
sidebar, enable *Notebook access*) so it is not saved into the notebook file.
Falls back to a hidden prompt if the secret is missing.

In [ ]:
from roboflow import Roboflow
try:
    from google.colab import userdata
    KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    import getpass
    KEY = getpass.getpass('Roboflow API key: ')
rf = Roboflow(api_key=KEY)

## Datasets

Two sources, chosen for **regional** fit rather than size. Western sets (KITTI
etc.) contain no microbus and no tuk-tuk, which are exactly the vehicles this
camera sees:

| source | why |
|---|---|
| `vehicle-detection-l6whj` | has `Threewheel` (tuk-tuk) and `minibus` |
| `vehicle-models-y8tzo` | Gulf traffic-authority taxonomy: `micro-bus`, `hgv-*`, `lgv-*` |

**Every dataset gets its real folders inspected before use.** A third candidate
(`vehicle-class-cr0z4`) advertised `LCV, Truck, Bus, Car...` on Universe *and via
the API*, but its actual folders were lines of the Roboflow README
(`'* collect & organize images'`, 1269 images). Training on it would have
produced excellent metrics for classifying README paragraphs. Universe metadata
is not trustworthy; the folder listing is.

In [ ]:
import glob, os, collections

def fetch(workspace, project, dest):
    p = rf.workspace(workspace).project(project)
    v = [x.version for x in p.versions()][-1]
    loc = p.version(v).download('folder', location=dest).location
    folders = collections.Counter(
        os.path.basename(os.path.dirname(f)) for f in glob.glob(loc + '/*/*/*'))
    print(f'\n{project} v{v} -> {loc}')
    for k, n in folders.most_common(40):
        print(f'   {k[:44]:44s} {n}')
    return loc

RF2 = fetch('vehicle-edlra', 'vehicle-detection-l6whj', '/content/rf2')
RF3 = fetch('atul-arun-kanse-njfby', 'vehicle-models-y8tzo', '/content/rf3')

## Map source classes onto the mentor taxonomy

Anything not in `MAP` is **dropped and reported**, which is deliberate:

* `fire engine` — genuinely ambiguous between D and F
* multi-label folders (`car motorbike`, `bus threewheel`) — two vehicles in one
  image cannot supply a single label

`lgv` **is** mapped, to C. It was excluded in v1 as "ambiguous between C and V",
which was wrong — the Gulf taxonomy has a separate `lgv-van` for vans, so plain
`lgv` is the generic light goods vehicle. Excluding it starved C (478 images vs
D's 1132) and the model learned *truck-shaped => heavy*, sweeping 13 of 19 real
light trucks into D.

In [ ]:
import shutil, random

MAP = {
    # A - private car
    'car': 'A', 'suv': 'A', 'jeep': 'A', 'taxi': 'A', 'sedan': 'A',
    'family sedan': 'A', 'racing car': 'A', 'passenger-car': 'A',
    # C - light truck / light commercial
    'truck': 'C', 'pickup': 'C', 'lgv-pickup': 'C', 'lgv': 'C',
    # D - heavy truck
    'heavy truck': 'D', 'hgv-2-axel-rigid': 'D',
    'hgv-3-or-more-axel-rigid': 'D', 'hgv-3-or-more-axel-articulated': 'D',
    # E - bus (incl. microbus, the dominant vehicle on this road)
    'bus': 'E', 'minibus': 'E', 'micro-bus': 'E', 'medium-bus': 'E',
    'large-labor-bus': 'E', 'rta-bus': 'E', 'school-bus': 'E',
    'company-tourist-bus': 'E',
    # G - two/three wheelers (three-wheeler = tuk-tuk)
    'motorbike': 'G', 'motorcycle': 'G', 'motor-cycle': 'G', 'bike': 'G',
    'bicycle': 'G', 'e-scooter': 'G', 'threewheel': 'G', 'three-wheeler': 'G',
    # V - van
    'van': 'V', 'lgv-van': 'V',
}
DST = '/content/its7'
shutil.rmtree(DST, ignore_errors=True)

def key(f):
    return os.path.basename(os.path.dirname(f)).strip().strip('\'"').lower()

files = glob.glob(RF2 + '/*/*/*') + glob.glob(RF3 + '/*/*/*')
kept = 0
for i, f in enumerate(files):
    code = MAP.get(key(f))
    if not code:
        continue
    split = 'train' if '/train/' in f.replace(os.sep, '/') else 'val'
    out = os.path.join(DST, split, code)
    os.makedirs(out, exist_ok=True)
    shutil.copy(f, os.path.join(out, f'{i}_{os.path.basename(f)}'))
    kept += 1

dropped = collections.Counter(key(f) for f in files if key(f) not in MAP)
print('kept', kept, 'images')
print('DROPPED (>=40 images, deliberately):',
      sorted((k, n) for k, n in dropped.items() if n >= 40))

## Balance

Unbalanced, A outnumbers V ~20:1 and the model learns to answer "car" and score
60%. Capping the majority classes is cruder than class-weighted loss but it is
transparent, and with a rare class this small the honest fix is more V images,
not a cleverer loss.

In [ ]:
CAP = 1600
random.seed(0)
for c in sorted(os.listdir(DST + '/train')):
    fs = glob.glob(f'{DST}/train/{c}/*')
    for q in random.sample(fs, max(len(fs) - CAP, 0)):
        os.remove(q)

for split in ('train', 'val'):
    print(split, {c: len(os.listdir(f'{DST}/{split}/{c}'))
                  for c in sorted(os.listdir(f'{DST}/{split}'))})

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s-cls.pt')
model.train(
    data=DST, epochs=30, imgsz=128, batch=64, device=0,
    name='its7class', patience=10, cos_lr=True,
    # A mirrored vehicle is still that vehicle (unlike PLATES, where mirroring
    # reverses reading order - see tools/train_plates.py).
    fliplr=0.5, flipud=0.0, degrees=8.0,
    # Deployment crops are small, blurry and JPEG-damaged; training on clean
    # photos alone produces a model that meets nothing like them.
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.5, erasing=0.3,
)

## Save the weights NOW

Colab runtimes are reclaimed without warning and session storage goes with them —
this has already cost one training run at epoch 23 of 30. Download before doing
anything else.

In [ ]:
BEST = '/content/runs/classify/its7class/weights/best.pt'
shutil.copy(BEST, '/content/vehicle_cls.pt')
print('%.1f MB' % (os.path.getsize('/content/vehicle_cls.pt') / 1e6),
      '| classes:', YOLO(BEST).names)
from google.colab import files
files.download('/content/vehicle_cls.pt')

## The test that actually decides it

Upload `its_vehicle_cls.zip` (produced by `tools/train_vehicle_classes.py
--prepare` from your own labelled crops) using the **Files** panel on the left,
then run the cell below.

These crops come from the deployment camera and the model has never seen any of
them, so all of them are a genuine held-out test. Compare against the size
heuristic's **0.500** on the same vehicles.

Per-**vehicle** is the number that matters: the pipeline votes a class across
every crop of a track, so a single bad frame does not decide the answer.

In [ ]:
import zipfile

zips = glob.glob('/content/*vehicle_cls*.zip')
assert zips, 'Upload its_vehicle_cls.zip via the Files panel first'
shutil.rmtree('/content/mycam', ignore_errors=True)
zipfile.ZipFile(zips[0]).extractall('/content/mycam')

best = YOLO(BEST)
paths = sorted(glob.glob('/content/mycam/*/*/*'))
truth = [p.replace(os.sep, '/').split('/')[-2] for p in paths]
pred = [best.names[int(r.probs.top1)]
        for r in best.predict(paths, verbose=False, imgsz=128)]

print('=== ON THE DEPLOYMENT CAMERA (never seen in training) ===')
print('CROPS    n=%d  acc=%.3f' %
      (len(paths), sum(t == q for t, q in zip(truth, pred)) / len(paths)))

byv = collections.defaultdict(list)
for p, t, q in zip(paths, truth, pred):
    byv[os.path.basename(p).rsplit('_', 1)[0]].append((t, q))
voted = {k: (v[0][0], collections.Counter(q for _, q in v).most_common(1)[0][0])
         for k, v in byv.items()}
print('VEHICLES n=%d  acc=%.3f   (size heuristic on the same set: 0.500)' %
      (len(voted), sum(t == q for t, q in voted.values()) / len(voted)))

classes = sorted({t for t, _ in voted.values()})
for c in classes:
    hits = [(t, q) for t, q in voted.values() if t == c]
    print('   %s  n=%3d  acc=%.3f  mistaken_for=%s' % (
        c, len(hits), sum(t == q for t, q in hits) / max(len(hits), 1),
        collections.Counter(q for t, q in hits if t != q).most_common(3)))